In [1]:
"""
CarbonTatva AI — Model: Linear Regression + Ridge
==================================================
Last-year data: Mar 2025 – Feb 2026
Split:  Train = Mar–Sep 2025  |  Val = Oct–Nov 2025  |  Test = Dec 2025–Feb 2026

Key fixes:
  - Chronological split (train→val→test)
  - Lag features built on FULL data so lag672 at Mar-2025
    correctly pulls from Sep-2024 (not NaN)
  - Then filter to last-year AFTER building features
"""

import os, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings('ignore')
os.makedirs('outputs_ridge_lastyear', exist_ok=True)

# ─────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────
DATA_FILE       = 'final_merged_data.xlsx'
MARKETS         = ['GDAM', 'DAM', 'RTM']
SLOTS_DAY       = 96
LAST_YEAR_START = '2025-03-01'   # filter window start
TRAIN_END       = '2025-09-30'   # ← chronological: train ends Sep
VAL_END         = '2025-11-30'   # ← val ends Nov, test = Dec onwards
SEED            = 42
np.random.seed(SEED)

# ─────────────────────────────────────────────────────────────────
# FEATURE ENGINEERING
# Build on FULL dataframe first so lag/MA features are correct,
# then filter to last year.
# ─────────────────────────────────────────────────────────────────
def build_features(df):
    df = df.copy()
    d = pd.to_datetime(df['Delivery Date'], dayfirst=True)
    t = df['Time Period'].str.split('-').str[0]
    df['datetime'] = pd.to_datetime(d.astype(str) + ' ' + t)
    df = df.sort_values('datetime').reset_index(drop=True)

    df.drop(columns=['apparent_temperature (°C)',
                     'sunshine_duration (s)',
                     'precipitation (mm)'], errors='ignore', inplace=True)

    df['hour']  = df['datetime'].dt.hour
    df['dow']   = df['datetime'].dt.dayofweek
    df['month'] = df['datetime'].dt.month

    df['hour_sin']  = np.sin(2 * np.pi * df['hour']  / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['hour']  / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['dow_sin']   = np.sin(2 * np.pi * df['dow']   / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df['dow']   / 7)
    df['is_weekend'] = (df['dow'] >= 5).astype(int)

    try:
        import holidays
        india_h = holidays.India(years=range(2022, 2027))
        df['holiday_flag'] = df['datetime'].dt.date.apply(
            lambda d: 1 if d in india_h else 0)
    except ImportError:
        df['holiday_flag'] = 0

    # AR features on FULL data — critical for correct lag values
    p = df['Price (Rs./MWh)']
    df['price_lag96']  = p.shift(96)
    df['price_lag672'] = p.shift(672)
    df['ma3d']  = p.shift(1).rolling(288).mean()
    df['ma7d']  = p.shift(1).rolling(672).mean()
    df['ma14d'] = p.shift(1).rolling(1344).mean()
    df['price_diff1d'] = p.shift(96)  - p.shift(192)
    df['price_diff1w'] = p.shift(672) - p.shift(1344)

    # Price level features
    df['price_level_ratio'] = df['price_lag96'] / (df['ma7d'] + 1e-6)
    df['rel_lag672']        = df['price_lag672'] / (df['ma7d'] + 1e-6)

    # Weather
    df['temp']    = df['temperature_2m (°C)']
    df['temp_sq'] = df['temp'] ** 2
    df['temp_cb'] = df['temp'] ** 3
    df['radiation']            = df['shortwave_radiation (W/m²)']
    df['radiation_x_hour_sin'] = df['radiation'] * df['hour_sin']
    df['radiation_x_hour_cos'] = df['radiation'] * df['hour_cos']
    df['radiation_sq']         = df['radiation'] ** 2
    df['wind_speed']  = df['wind_speed_100m (km/h)']
    df['wind_sq']     = df['wind_speed'] ** 2
    df['cloud_cover'] = df['cloud_cover (%)']
    df['humidity']    = df['relative_humidity_2m (%)']
    df['rain']        = df['rain (mm)']
    df['temp_x_hour_sin']   = df['temp'] * df['hour_sin']
    df['temp_x_is_weekend'] = df['temp'] * df['is_weekend']
    df['wind_x_radiation']  = df['wind_speed'] * df['radiation']

    return df


FEATURES = [
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'dow_sin', 'dow_cos', 'is_weekend', 'holiday_flag',
    'price_lag96', 'price_lag672', 'ma3d', 'ma7d', 'ma14d',
    'price_diff1d', 'price_diff1w',
    'price_level_ratio', 'rel_lag672',
    'temp', 'temp_sq', 'temp_cb',
    'radiation', 'radiation_sq', 'radiation_x_hour_sin', 'radiation_x_hour_cos',
    'wind_speed', 'wind_sq', 'cloud_cover', 'humidity', 'rain',
    'temp_x_hour_sin', 'temp_x_is_weekend', 'wind_x_radiation',
]

# ─────────────────────────────────────────────────────────────────
# UTILITIES
# ─────────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, label=''):
    mae  = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mask = y_true > 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    print(f"    {label:20s}  MAE={mae:7.1f}  RMSE={rmse:7.1f}  MAPE={mape:.2f}%")
    return {'mae': mae, 'rmse': rmse, 'mape': mape}


def save_pred_plot(y_true, y_pred, title, path):
    fig, axes = plt.subplots(3, 1, figsize=(16, 12))
    fig.patch.set_facecolor('#0f1117')
    for ax in axes:
        ax.set_facecolor('#1a1d27')
        ax.tick_params(colors='#aaa')
    n = min(800, len(y_true))
    axes[0].plot(y_true[:n], color='#aaa', lw=0.8, label='Actual')
    axes[0].plot(y_pred[:n], color='#8e44ad', lw=0.8, label='Predicted')
    axes[0].set_title(f'{title} — Test Set', color='white', fontsize=12)
    axes[0].legend(fontsize=9)
    axes[0].set_ylabel('Price (Rs./MWh)', color='#aaa')
    n2 = SLOTS_DAY * 3
    axes[1].plot(y_true[:n2], color='#aaa', lw=1.5, label='Actual')
    axes[1].plot(y_pred[:n2], color='#8e44ad', lw=1.5, ls='--', label='Predicted')
    axes[1].set_title(f'{title} — First 3 Days', color='white', fontsize=12)
    axes[1].legend(fontsize=9)
    axes[1].set_ylabel('Price (Rs./MWh)', color='#aaa')
    axes[2].scatter(y_true[:3000], y_pred[:3000], s=2, alpha=0.3, color='#8e44ad')
    lim = max(y_true[:3000].max(), y_pred[:3000].max())
    axes[2].plot([0, lim], [0, lim], 'r--', lw=1.5, label='Perfect')
    axes[2].set_title(f'{title} — Actual vs Predicted', color='white', fontsize=12)
    axes[2].set_xlabel('Actual', color='#aaa')
    axes[2].set_ylabel('Predicted', color='#aaa')
    axes[2].legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close()


# ─────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────
print("=" * 60)
print("MODEL: Ridge Regression (Last Year Data)")
print(f"Period: {LAST_YEAR_START} onwards")
print(f"Split:  Train≤{TRAIN_END}  |  Val≤{VAL_END}  |  Test>Val")
print("=" * 60)

sheets = pd.read_excel(DATA_FILE, sheet_name=None)
all_metrics = {}

for mkt in MARKETS:
    print(f"\n{'─'*50}\nMarket: {mkt}\n{'─'*50}")

    # Build features on FULL data first
    df_full = build_features(sheets[mkt])
    df_full = df_full.dropna(subset=FEATURES + ['Price (Rs./MWh)']).copy()

    # NOW filter to last year
    df = df_full[df_full['datetime'] >= LAST_YEAR_START].copy()
    print(f"  Rows: {len(df):,}  ({df['datetime'].min().date()} → {df['datetime'].max().date()})")

    train = df[df['datetime'] <= TRAIN_END]
    val   = df[(df['datetime'] > TRAIN_END) & (df['datetime'] <= VAL_END)]
    test  = df[df['datetime'] > VAL_END]
    print(f"  Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")
    print(f"  Price mean — Train: {train['Price (Rs./MWh)'].mean():.0f}  Val: {val['Price (Rs./MWh)'].mean():.0f}  Test: {test['Price (Rs./MWh)'].mean():.0f}")

    X_tr = train[FEATURES].values;  y_tr = train['Price (Rs./MWh)'].values
    X_va = val[FEATURES].values;    y_va = val['Price (Rs./MWh)'].values
    X_te = test[FEATURES].values;   y_te = test['Price (Rs./MWh)'].values

    scaler  = StandardScaler()
    X_tr_s  = scaler.fit_transform(X_tr)
    X_va_s  = scaler.transform(X_va)
    X_te_s  = scaler.transform(X_te)

    # Alpha grid search on validation
    print("  Tuning alpha...")
    best_alpha, best_mae = None, float('inf')
    for alpha in [0.001, 0.01, 0.1, 1.0, 10.0, 50.0, 100.0, 500.0, 1000.0, 5000.0, 10000.0]:
        m = Ridge(alpha=alpha, max_iter=10000)
        m.fit(X_tr_s, y_tr)
        vm = mean_absolute_error(y_va, m.predict(X_va_s))
        if vm < best_mae:
            best_mae, best_alpha = vm, alpha
    print(f"  Best alpha: {best_alpha}  |  Val MAE: {best_mae:.1f}")

    # Final model: refit on train+val combined
    X_tv = np.vstack([X_tr_s, X_va_s])
    y_tv = np.concatenate([y_tr, y_va])
    final_model = Ridge(alpha=best_alpha, max_iter=10000)
    final_model.fit(X_tv, y_tv)

    val_pred  = np.clip(final_model.predict(X_va_s), 0, 20000)
    test_pred = np.clip(final_model.predict(X_te_s), 0, 20000)

    print("  Results:")
    m_val  = compute_metrics(y_va, val_pred,  'Validation')
    m_test = compute_metrics(y_te, test_pred, 'Test')
    all_metrics[mkt] = {'val': m_val, 'test': m_test}

    save_pred_plot(y_te, test_pred,
                   f'Ridge — {mkt}',
                   f'outputs_ridge_lastyear/ridge_pred_{mkt}.png')

    # Coefficient plot
    idx = np.argsort(np.abs(final_model.coef_))
    bar_c = ['#e74c3c' if c > 0 else '#2980b9' for c in final_model.coef_[idx]]
    fig, ax = plt.subplots(figsize=(11, 10))
    fig.patch.set_facecolor('#0f1117'); ax.set_facecolor('#1a1d27')
    ax.barh([FEATURES[i] for i in idx], final_model.coef_[idx], color=bar_c, alpha=0.85)
    ax.axvline(0, color='white', lw=0.8, ls='--')
    ax.set_title(f'Ridge Coefficients — {mkt}\nRed=+price  Blue=-price', color='white')
    ax.tick_params(colors='#aaa')
    plt.tight_layout()
    plt.savefig(f'outputs_ridge_lastyear/ridge_coefs_{mkt}.png',
                dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close()

    with open(f'outputs_ridge_lastyear/ridge_{mkt}.pkl', 'wb') as f:
        pickle.dump({'model': final_model, 'scaler': scaler,
                     'features': FEATURES, 'best_alpha': best_alpha}, f)
    print(f"  Saved: outputs_ridge_lastyear/ridge_{mkt}.pkl")

# Summary
print(f"\n{'='*60}\nSUMMARY — Ridge Regression (Last Year)\n{'='*60}")
rows = []
for mkt, res in all_metrics.items():
    for split, m in res.items():
        rows.append({'Market': mkt, 'Split': split,
                     'MAE': round(m['mae'],1), 'RMSE': round(m['rmse'],1),
                     'MAPE%': round(m['mape'],2)})
summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))
summary_df.to_csv('outputs_ridge_lastyear/metrics.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('#0f1117')
fig.suptitle('Ridge — Test MAE by Market (Last Year)', color='white', fontsize=14)
colors = {'GDAM':'#e74c3c','DAM':'#2980b9','RTM':'#27ae60'}
for ax, mkt in zip(axes, MARKETS):
    ax.set_facecolor('#1a1d27')
    if mkt in all_metrics:
        mae = all_metrics[mkt]['test']['mae']
        ax.bar([mkt], [mae], color=colors[mkt], alpha=0.85)
        ax.text(0, mae+5, f'Rs.{mae:.0f}', ha='center', color='white', fontsize=12, fontweight='bold')
    ax.set_title(mkt, color='white')
    ax.set_ylabel('MAE (Rs./MWh)', color='#aaa')
    ax.tick_params(colors='#aaa')
plt.tight_layout()
plt.savefig('outputs_ridge_lastyear/summary_chart.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.close()
print("\n✅ Done! Check outputs_ridge_lastyear/")


MODEL: Ridge Regression (Last Year Data)
Period: 2025-03-01 onwards
Split:  Train≤2025-09-30  |  Val≤2025-11-30  |  Test>Val

──────────────────────────────────────────────────
Market: GDAM
──────────────────────────────────────────────────
  Rows: 35,040  (2025-03-01 → 2026-02-28)
  Train: 20,449 | Val: 5,856 | Test: 8,735
  Price mean — Train: 4657  Val: 3579  Test: 4256
  Tuning alpha...
  Best alpha: 100.0  |  Val MAE: 530.3
  Results:
    Validation            MAE=  470.5  RMSE=  730.8  MAPE=16.99%
    Test                  MAE=  483.3  RMSE=  811.0  MAPE=10.53%
  Saved: outputs_ridge_lastyear/ridge_GDAM.pkl

──────────────────────────────────────────────────
Market: DAM
──────────────────────────────────────────────────
  Rows: 35,040  (2025-03-01 → 2026-02-28)
  Train: 20,449 | Val: 5,856 | Test: 8,735
  Price mean — Train: 4197  Val: 2867  Test: 3782
  Tuning alpha...
  Best alpha: 100.0  |  Val MAE: 696.2
  Results:
    Validation            MAE=  556.6  RMSE=  855.6  MAPE=48.